# Questions for all task objects

Generate one source-grounded Chinese question for each task in documents with nonzero workbook `task_count`. Profile-specific workbook rows determine eligibility; mirrored task records are counted once. Distinct corpus profiles remain distinct cases.

Run this notebook first, then `05.application.rag.task.ipynb`. Question authoring uses the configured RAG model at **172.27.74.16:11434**. It needs the saved `_tasks.json` documents but does not need a vector index. Source references remain reviewer metadata. The shared bank is published only after every selected task succeeds.

In [ ]:
# | notest
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJ_ROOT = Path(os.getenv("PROJ_ROOT", Path.cwd())).resolve()
while not (PROJ_ROOT / "pyproject.toml").is_file() and PROJ_ROOT != PROJ_ROOT.parent:
    PROJ_ROOT = PROJ_ROOT.parent
if not (PROJ_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Set PROJ_ROOT to the ribosome project directory")
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
load_dotenv(PROJ_ROOT / ".env", override=False)
from ribosome.ollama_config import OLLAMA_EMBED_HOST, OLLAMA_RAG_HOST, OLLAMA_TRUST_ENV

RAG_MODEL = os.getenv("OLLAMA_RAG_MODEL") or os.getenv("OLLAMA_SUMMARY_MODEL", "")
REQUEST_TIMEOUT = 300
BANK_PATH = PROJ_ROOT / "db/task_rag_tests/question_with_tasks/bank.json"
HEADERS = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]} if os.getenv("OLLAMA_API_KEY") else {}

from utils.task_rag_questions import (
    TaskQuestionAuthor, enumerate_task_cases, file_sha256, select_task_sources,
    validate_bank,
)
WORKBOOK = PROJ_ROOT / "proj_config/coding_categories/coding_manuals_markdown.all.xlsx"
CORPUS_DIRS = {profile: PROJ_ROOT / f"assets/code-docs.{profile}/.md" for profile in ("4.0", "5.0")}
AUTHORING_DIRECTORY = BANK_PATH.parent / "authored"
AUTHORING_CONCURRENCY = 2
PREVIEW_LIMIT = 20


## Select documents and enumerate tasks

The actual saved task objects determine coverage. Count differences are displayed for review; all tasks in each selected document are included. Missing, malformed, ambiguous, or empty task sources stop authoring with an explicit error.

In [ ]:
# | notest
selections = select_task_sources(PROJ_ROOT, WORKBOOK, CORPUS_DIRS)
task_cases, document_counts = enumerate_task_cases(PROJ_ROOT, selections)
display(pd.DataFrame(document_counts))
print(f"Selected {len(document_counts)} documents and {len(task_cases)} unique tasks")
if any(not row["count_matches"] for row in document_counts):
    print("Workbook/task count differences detected; every actual task will be included.")
selection_manifest = dict(
    workbook=str(WORKBOOK.relative_to(PROJ_ROOT)), workbook_sha256=file_sha256(WORKBOOK),
    documents=document_counts,
)


## Author and publish the complete bank

Only exact task content and source-derived context are supplied to the author. Stored summaries are excluded. Responses require one Chinese question and supporting quotations from the table. Successful checkpoints are reused; failed responses are retried on rerun. Each malformed response receives one corrective retry. A failure or interruption preserves the previous complete bank.

In [ ]:
# | notest
from ollama import AsyncClient
if not RAG_MODEL:
    raise ValueError("Configure OLLAMA_RAG_MODEL or OLLAMA_SUMMARY_MODEL")
authoring_client = AsyncClient(host=OLLAMA_RAG_HOST, headers=HEADERS,
                              timeout=REQUEST_TIMEOUT, trust_env=OLLAMA_TRUST_ENV)
try:
    author = TaskQuestionAuthor(authoring_client, RAG_MODEL, OLLAMA_RAG_HOST, AUTHORING_DIRECTORY)
    report = await author.publish(task_cases, document_counts, BANK_PATH,
                                  selection=selection_manifest, concurrency=AUTHORING_CONCURRENCY)
finally:
    await authoring_client._client.aclose()
display(pd.DataFrame(report["failures"]))
if not report["published"]:
    raise RuntimeError(f"{len(report['failures'])} task questions failed; previous bank preserved. Rerun this cell to retry.")
bank = validate_bank(report["bank"])
print(f"Published {len(bank['cases'])} questions: {BANK_PATH}")


## Bounded question preview

In [ ]:
# | notest
display(pd.DataFrame([
    dict(case_id=case["id"], category=case["category"], profile=case["profile"],
         task_id=case["task_id"], section=" → ".join(case["heading_path"]),
         source=case["source_md"], line_start=case["references"][0]["line_start"],
         line_end=case["references"][0]["line_end"], question=case["question"])
    for case in bank["cases"][:PREVIEW_LIMIT]
]))
print(f"Showing up to {PREVIEW_LIMIT} of {bank['task_count']} task questions")
